# Prepare CIFAR-100 EmbeddingGemma 2 vectors

Download the CIFAR-100 training set, preserve both its 100 fine labels and 20 coarse superclass labels, and embed the images with Google's google/embeddinggemma-2 multimodal model. The default processes all 50,000 training images and saves normalized 768-dimensional vectors, both labels, source indices, the taxonomy, and a manifest under data/cifar100/cache/.

Install notebook dependencies in the active kernel first:

```python
%pip install -U sentence-transformers transformers torchvision pillow
```

Model weights are fetched on the first run; access may require accepting the model terms on Hugging Face. See Google's [multimodal EmbeddingGemma 2 guide](https://ai.google.dev/gemma/docs/embeddinggemma/multimodal-embeddinggemma-with-sentence-transformers).

In [1]:
from pathlib import Path

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "imste" / "__init__.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository from the notebook working directory.")

DATA_DIR = REPO_ROOT / "data" / "cifar100"
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_ROOT)
print("CIFAR-100 data:", DATA_DIR)


Repository: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree
CIFAR-100 data: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar100


## Sampling and encoder settings

CIFAR-100 has 500 training images for each of its 100 fine classes, so SAMPLES_PER_FINE_CLASS = 500 embeds the full training split. Lower it for a quicker pilot. The encoder loads its image and text components but skips the unused audio encoder. Images are resized to 240 × 240 before encoding.

In [2]:
import hashlib
import json
import pickle
import time

import numpy as np
import torch
from PIL import Image
from torchvision.datasets import CIFAR100

MODEL_ID = "google/embeddinggemma-2"
RANDOM_STATE = 42
SAMPLES_PER_FINE_CLASS = 500  # 500 means all 50,000 CIFAR-100 training images.
ENCODER_BATCH_SIZE = 32
IMAGE_SIZE = 240
FORCE_REEMBED = False

if torch.cuda.is_available():
    DEVICE = "cuda"
    TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    TORCH_DTYPE = torch.float32  # EmbeddingGemma 2 activations do not support float16.
else:
    DEVICE = "cpu"
    TORCH_DTYPE = torch.float32

MODEL_TAG = MODEL_ID.rsplit("/", 1)[-1].replace("-", "_")
EMBEDDING_CACHE = CACHE_DIR / (
    f"cifar100_train_{SAMPLES_PER_FINE_CLASS}_per_fine_class_seed"
    f"{RANDOM_STATE}_{MODEL_TAG}.npz"
)
MANIFEST_PATH = CACHE_DIR / "embeddinggemma2_cifar100_manifest.json"

print(f"Embedding model: {MODEL_ID} | device={DEVICE} | dtype={TORCH_DTYPE}")
print(f"Sample: {SAMPLES_PER_FINE_CLASS:,} images per fine class")
print(f"Embedding cache: {EMBEDDING_CACHE}")


Embedding model: google/embeddinggemma-2 | device=mps | dtype=torch.float32
Sample: 500 images per fine class
Embedding cache: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar100/cache/cifar100_train_500_per_fine_class_seed42_embeddinggemma_2.npz


In [3]:
train_data = CIFAR100(root=str(RAW_DIR), train=True, download=True)
raw_train_path = RAW_DIR / "cifar-100-python" / "train"
metadata_path = RAW_DIR / "cifar-100-python" / "meta"

with raw_train_path.open("rb") as file:
    raw_train = pickle.load(file, encoding="latin1")
with metadata_path.open("rb") as file:
    raw_metadata = pickle.load(file, encoding="latin1")

fine_names = list(raw_metadata["fine_label_names"])
coarse_names = list(raw_metadata["coarse_label_names"])
all_fine_labels = np.asarray(raw_train["fine_labels"], dtype=np.int64)
all_coarse_labels = np.asarray(raw_train["coarse_labels"], dtype=np.int64)
if not np.array_equal(all_fine_labels, np.asarray(train_data.targets, dtype=np.int64)):
    raise ValueError("The fine labels in torchvision and the CIFAR-100 archive do not match.")
if len(train_data.data) != len(all_fine_labels) or len(all_fine_labels) != len(all_coarse_labels):
    raise ValueError("The CIFAR-100 images and labels have inconsistent lengths.")

n_fine = len(fine_names)
n_coarse = len(coarse_names)
if n_fine != 100 or n_coarse != 20:
    raise ValueError(f"Expected 100 fine classes and 20 superclasses; got {n_fine} and {n_coarse}.")
if not 1 <= SAMPLES_PER_FINE_CLASS <= 500:
    raise ValueError("SAMPLES_PER_FINE_CLASS must be between 1 and 500.")

fine_counts = np.bincount(all_fine_labels, minlength=n_fine)
if not np.all(fine_counts == 500):
    raise ValueError("Expected 500 training images in each CIFAR-100 fine class.")

fine_to_coarse = np.full(n_fine, -1, dtype=np.int64)
for fine_id in range(n_fine):
    parents = np.unique(all_coarse_labels[all_fine_labels == fine_id])
    if len(parents) != 1:
        raise ValueError(f"Fine class {fine_names[fine_id]!r} maps to multiple superclasses.")
    fine_to_coarse[fine_id] = parents[0]

coarse_counts = np.bincount(fine_to_coarse, minlength=n_coarse)
if not np.all(coarse_counts == 5):
    raise ValueError("Expected exactly five fine classes in each superclass.")

print(f"Downloaded {len(train_data.data):,} training images.")
print(f"Fine classes: {n_fine} | coarse superclasses: {n_coarse}")
print("Images per superclass:", {coarse_names[i]: int(np.sum(all_coarse_labels == i)) for i in range(n_coarse)})


100%|██████████| 169M/169M [03:27<00:00, 814kB/s]  


Downloaded 50,000 training images.
Fine classes: 100 | coarse superclasses: 20
Images per superclass: {'aquatic_mammals': 2500, 'fish': 2500, 'flowers': 2500, 'food_containers': 2500, 'fruit_and_vegetables': 2500, 'household_electrical_devices': 2500, 'household_furniture': 2500, 'insects': 2500, 'large_carnivores': 2500, 'large_man-made_outdoor_things': 2500, 'large_natural_outdoor_scenes': 2500, 'large_omnivores_and_herbivores': 2500, 'medium_mammals': 2500, 'non-insect_invertebrates': 2500, 'people': 2500, 'reptiles': 2500, 'small_mammals': 2500, 'trees': 2500, 'vehicles_1': 2500, 'vehicles_2': 2500}


In [4]:
from tqdm.auto import tqdm

if EMBEDDING_CACHE.is_file() and not FORCE_REEMBED:
    with np.load(EMBEDDING_CACHE, allow_pickle=False) as saved:
        X = saved["embeddings"]
        fine_labels = saved["fine_labels"]
        coarse_labels = saved["coarse_labels"]
        sample_indices = saved["indices"]
        saved_model_id = str(saved["model_id"].item())
    if saved_model_id != MODEL_ID:
        raise ValueError("The cache model ID does not match MODEL_ID.")
    print("Loaded cached image vectors:", EMBEDDING_CACHE)
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sample_indices = np.concatenate(
        [
            rng.choice(
                np.flatnonzero(all_fine_labels == fine_id),
                SAMPLES_PER_FINE_CLASS,
                replace=False,
            )
            for fine_id in range(n_fine)
        ]
    )
    rng.shuffle(sample_indices)
    fine_labels = all_fine_labels[sample_indices]
    coarse_labels = all_coarse_labels[sample_indices]

    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(
        MODEL_ID,
        device=DEVICE,
        model_kwargs={"torch_dtype": TORCH_DTYPE},
        config_kwargs={"audio_config": None},
    )
    X = None
    started = time.perf_counter()
    for start in tqdm(
        range(0, len(sample_indices), ENCODER_BATCH_SIZE),
        desc="Embedding CIFAR-100 images",
    ):
        batch_indices = sample_indices[start : start + ENCODER_BATCH_SIZE]
        image_inputs = []
        for index in batch_indices:
            image = Image.fromarray(train_data.data[int(index)]).convert("RGB")
            image = image.resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                resample=Image.Resampling.BICUBIC,
            )
            image_inputs.append({"image": image})
        batch_vectors = model.encode(
            image_inputs,
            batch_size=len(image_inputs),
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        if X is None:
            X = np.empty(
                (len(sample_indices), batch_vectors.shape[1]),
                dtype=np.float32,
            )
        X[start : start + len(batch_indices)] = batch_vectors

    embedding_seconds = time.perf_counter() - started
    if X is None or not np.isfinite(X).all():
        raise ValueError("EmbeddingGemma 2 returned empty or non-finite image vectors.")
    np.savez_compressed(
        EMBEDDING_CACHE,
        embeddings=X,
        fine_labels=fine_labels,
        coarse_labels=coarse_labels,
        indices=sample_indices,
        fine_names=np.asarray(fine_names),
        coarse_names=np.asarray(coarse_names),
        fine_to_coarse=fine_to_coarse,
        model_id=np.asarray(MODEL_ID),
    )
    print(f"Encoded {len(X):,} images in {embedding_seconds / 60:.1f} min")
    print("Saved image vectors:", EMBEDDING_CACHE)

if X.ndim != 2 or X.shape[0] != len(fine_labels) or len(fine_labels) != len(coarse_labels):
    raise ValueError(f"Unexpected vector/label shapes: {X.shape}, {fine_labels.shape}, {coarse_labels.shape}")
if not np.isfinite(X).all():
    raise ValueError("The saved image vectors contain NaN or infinite values.")
vector_fingerprint = hashlib.sha256(
    np.ascontiguousarray(X).view(np.uint8)
).hexdigest()
manifest = {
    "dataset": "CIFAR-100 train",
    "model_id": MODEL_ID,
    "samples_per_fine_class": SAMPLES_PER_FINE_CLASS,
    "random_state": RANDOM_STATE,
    "image_size": IMAGE_SIZE,
    "embedding_file": EMBEDDING_CACHE.name,
    "shape": list(X.shape),
    "dtype": str(X.dtype),
    "vector_sha256": vector_fingerprint,
    "fine_label_names": fine_names,
    "coarse_label_names": coarse_names,
    "fine_to_coarse": fine_to_coarse.tolist(),
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + chr(10))

print("Embedding matrix:", X.shape, X.dtype)
print(f"Finite values: {np.isfinite(X).all()} | mean L2 norm: {np.linalg.norm(X, axis=1).mean():.4f}")
print("Fine class counts:", np.bincount(fine_labels, minlength=n_fine).tolist())
print("Superclass counts:", np.bincount(coarse_labels, minlength=n_coarse).tolist())
print("Manifest:", MANIFEST_PATH)


W1117 01:14:06.315000 43341 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/624 [00:00<?, ?it/s]

Embedding CIFAR-100 images:   0%|          | 0/1563 [00:00<?, ?it/s]

Encoded 50,000 images in 254.5 min
Saved image vectors: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar100/cache/cifar100_train_500_per_fine_class_seed42_embeddinggemma_2.npz
Embedding matrix: (50000, 768) float32
Finite values: True | mean L2 norm: 1.0000
Fine class counts: [500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
Superclass counts: [2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500, 2500]
Manifest: /Users/f.costa/Documents/Pro